# Quality benchmark — analysis

The generated pages hold the facts: [how it is run](../docs/benchmarks/quality-docs.md) and
[what came out](../docs/benchmarks/quality-results.md). This notebook is where the judgement
goes — what the summary numbers are made of, where the models disagree with the experts and
with each other, and what the photographs they get wrong look like.

The benchmark's primary question is **model against expert annotation**. Model-against-model
numbers are here to raise suspicions about the annotation, never to crown anything.

Run it after `python -m benchmarks --benchmark quality`; everything reads `results/quality/`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path('..').resolve() / 'results' / 'quality'
STORE = Path('..').resolve() / '.atlas_data'
WORTH = ['good', 'usable']

def per_image() -> pd.DataFrame:
    """Every model's answer about every photograph, one row each."""
    frames = []
    for path in sorted(RESULTS.glob('*/*.csv')):
        frame = pd.read_csv(path)
        frame['model'] = path.parent.name
        frame['dataset'] = path.stem
        frames.append(frame)
    answers = pd.concat(frames, ignore_index=True)
    answers['worth_measuring'] = answers['grade'].isin(WORTH)
    answers['said_worth'] = np.where(
        answers['verdict'].notna() & (answers['verdict'] != ''),
        answers['verdict'].isin(WORTH),
        answers['gradeable'] >= 0.5,
    )
    answers['right'] = answers['worth_measuring'] == answers['said_worth']
    return answers

def summaries() -> pd.DataFrame:
    rows = []
    for path in sorted(RESULTS.glob('*/*.json')):
        record = json.loads(path.read_text())
        s = record['summary']
        rows.append({
            'model': record['model'], 'dataset': record['dataset'],
            'processed': s['processed'], 'total': s['total'], 'complete': s['complete'],
            'coverage': s['coverage'],
            'accuracy': s['gradeable']['accuracy'], 'roc_auc': s['gradeable']['roc_auc'],
            'three_class': (s['three_class'] or {}).get('accuracy'),
        })
    return pd.DataFrame(rows)

answers = per_image()
scores = summaries()
scores


## 1. Coverage first

A model that declines a photograph has not got it wrong. Coverage is the share of a dataset a
model was willing to answer for at all, and it is read beside accuracy rather than folded into
it.


In [ ]:
answers.pivot_table(index=['model', 'dataset'], columns='outcome', values='key',
                    aggfunc='count').fillna(0).astype(int)


## 2. Accuracy and ranking are different questions

Accuracy depends on where a model's threshold sits, and one of these models' authors say
plainly that theirs does not transfer between datasets. The area under the ROC curve does not
depend on a threshold. A model can therefore rank photographs almost perfectly and still be
called inaccurate — a statement about the threshold, not about the model.


In [ ]:
wide = scores.pivot(index='dataset', columns='model', values=['accuracy', 'roc_auc'])
figure, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for axis, metric in zip(axes, ['accuracy', 'roc_auc']):
    wide[metric].plot.barh(ax=axis, xlim=(0.0, 1.0))
    axis.set_title(metric)
    axis.legend(fontsize=7)
figure.tight_layout()


### 2.1 The whole ROC curve, not one point on it

Each curve is one model on one dataset. Where two curves cross, the better model depends on
whether you would rather miss a bad photograph or throw away a good one.

Datasets whose reference contains no bad photographs are left out — a curve needs both
classes. They appear in section 6 instead, where what they measure is how much of a sound
dataset each model would discard.


In [ ]:
def curve(frame):
    """True and false positive rates at every threshold the model actually produced."""
    worth = frame['worth_measuring'].to_numpy()
    order = np.argsort(-frame['gradeable'].to_numpy())
    worth = worth[order]
    return (np.r_[0, np.cumsum(~worth) / max((~worth).sum(), 1)],
            np.r_[0, np.cumsum(worth) / max(worth.sum(), 1)])

graded = answers[answers['outcome'] == 'graded']
both = [d for d, f in graded.groupby('dataset') if f['worth_measuring'].nunique() == 2]
print('no ranking is defined on:', sorted(set(graded['dataset']) - set(both)))

figure, axes = plt.subplots(1, len(both), figsize=(4 * len(both), 4), squeeze=False)
for axis, dataset in zip(axes[0], sorted(both)):
    for model, frame in graded[graded['dataset'] == dataset].groupby('model'):
        axis.plot(*curve(frame), label=model, linewidth=1)
    axis.plot([0, 1], [0, 1], color='grey', linewidth=0.5)
    axis.set_title(dataset, fontsize=9)
    axis.set_xlabel('kept, though bad')
    axis.set_ylabel('kept, and worth measuring')
    axis.legend(fontsize=6)
figure.tight_layout()


## 3. The same photographs, grouped as the dataset groups them

The run scores a dataset whole and records where each photograph came from, so the grouping is
a question asked here rather than one the run had to be told. A camera or a split that behaves
differently from its neighbours is the kind of thing a single number hides — and if one turns
out to matter, it belongs in the generated results page too, not only here.


In [ ]:
by_group = graded.groupby(['model', 'dataset', 'subset', 'split'])['right'].agg(['mean', 'size'])
by_group.round(3)


## 4. Where the models disagree with each other

Two models agreeing is not two pieces of evidence when they learned from the same labels — and
three of these were fitted on EyeQ. Agreement with a model trained on other data is the more
informative number, and disagreement is where to go looking for a bad annotation.


In [ ]:
verdicts = graded.pivot_table(index=['dataset', 'key'], columns='model', values='said_worth')
models = list(verdicts.columns)
pd.DataFrame(
    [[(verdicts[a] == verdicts[b]).mean() for b in models] for a in models],
    index=models, columns=models,
).round(3)


## 5. Where the models disagree with the readers

Two of these datasets kept their readers apart, so the photographs the readers themselves
disagreed about can be separated from the ones they were sure of. A model that is wrong where
the humans disagreed is in different trouble from one that is wrong where they did not — and
the first kind is a reason to look again at the reference.


In [ ]:
def readers(entry) -> list[str]:
    return [part.split('=')[1] for part in str(entry).split(';') if '=' in part]

multi = graded[graded['readers'].fillna('') != ''].copy()
multi['unanimous'] = multi['readers'].map(lambda entry: len(set(readers(entry))) == 1)
multi.pivot_table(index=['model', 'dataset'], columns='unanimous', values='right').round(3)


## 6. What a model would throw away

[PAPILA](../docs/datasets/papila.md) publishes no quality grades. It is in this benchmark on
this repository's assumption that all 488 of its photographs are sound — one camera,
disc-centred, every frame outlined by two ophthalmologists. That makes it useless for asking
whether a model finds bad photographs, and the only place here that can answer a question every
user of these pipelines has: run this gate over a clean dataset, and how much of it disappears?

**The black bands are not the explanation.** PAPILA's retina fills the frame vertically, so the
square the store builds is nearly a fifth canvas — and a quality model judges the square it is
handed. The cell below trims those bands off and hands the model the photograph's own 4:3
rectangle instead. When this was last run the share kept moved *down*, not up: the square is the
familiar shape for models fitted on screening photographs, so the canvas is not what makes them
reject it.


In [ ]:
assumed = [d for d in graded['dataset'].unique() if d == 'papila']
if assumed:
    clean = graded[graded['dataset'].isin(assumed)].copy()
    # The column is empty for a model no pipeline gates on, and pandas reads what is left as
    # booleans or numbers depending on what it is concatenated with, so accept both spellings.
    carried = clean['carried_by_its_pipeline'].astype(str).str.lower()
    clean['kept_by_its_pipeline'] = carried.map({'true': 1.0, '1.0': 1.0, 'false': 0.0, '0.0': 0.0})
    display(clean.groupby('model')[['said_worth', 'kept_by_its_pipeline']].mean().round(3))


## 7. The photographs everything gets wrong

A table says a model scores 0.82. This is where someone finds out whether the 0.82 is two
populations, and whether one of them is a camera — or whether the grade itself is wrong.


In [ ]:
from PIL import Image

wrong = graded.groupby(['dataset', 'key'])['right'].mean()
hardest = wrong[wrong == 0].index[:8]

figure, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for axis, (dataset, key) in zip(axes.ravel(), hardest):
    axis.imshow(Image.open(STORE / dataset / '512' / 'images' / f'{key}.png'))
    said = graded[(graded['dataset'] == dataset) & (graded['key'] == key)]
    axis.set_title(f"{key}\ndataset says {said['grade'].iloc[0]}", fontsize=8)
    axis.axis('off')
figure.tight_layout()


## 8. What this benchmark cannot say

- **A model marked `unknown` is not cleared.** VascX publishes no list of what it trained on,
  and its shipped configuration names EyeQ for this model, so no out-of-sample claim can be
  made for it here.
- **The three-class comparison is not like for like.** The datasets' own grades were defined by
  different people for different purposes, and one of them never uses the middle class at all.
- **An assumed reference measures what a model discards, not how accurate it is.**
- **Nothing here says a model is best.** Read a row, not a column.
